# 02 — Exploração: Lotes e Veículos DETRAN/MG

**Expansão do scraping** — validar extração de lotes/veículos antes de persistir em massa.

**Objetivos:**
1. Inspecionar HTML da listagem `/lotes/lista-lotes/{id}/{ano}`
2. Entender paginação (8 lotes por página)
3. Construir e validar parser com `pandas`
4. Conferir qualidade (nulls, valores BRL, condições)
5. Validar módulo `detran_scraper` (`parse_lotes`, paginação)

**Fonte:** https://leilao.detran.mg.gov.br/

## 1. Setup e escolha de um edital

Reutilizamos a home para pegar um `url_detalhes` e então acessar a listagem de lotes.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from detran_scraper import DetranClient, parse_editais, parse_lotes, parse_lotes_max_page

BASE_URL = os.getenv("DETRAN_BASE_URL", "https://leilao.detran.mg.gov.br")

with DetranClient(base_url=BASE_URL) as client:
    home_html = client.fetch_home()
    editais = parse_editais(home_html, base_url=BASE_URL)

edital = editais[0]
lista_path = edital.url_detalhes.replace(BASE_URL, "")
print(f"Edital: {edital.numero_edital} | leilao_id={edital.leilao_id}")
print(f"Listagem: {lista_path}")

with DetranClient(base_url=BASE_URL) as client:
    lotes_html = client.fetch(lista_path)

print(f"HTML: {len(lotes_html):,} bytes")
print(f"Cards na página 1: {len(BeautifulSoup(lotes_html, 'lxml').select('div.card.listaLotes'))}")
print(f"Última página: {parse_lotes_max_page(lotes_html)}")

Edital: 1692/2026 | leilao_id=3416
Listagem: /lotes/lista-lotes/3416/2026
HTML: 57,960 bytes
Cards na página 1: 8
Última página: 14


## 2. Inspeção do card de lote

Cada veículo é um `div.card.listaLotes` com `id` = `lote_id`. Campos visíveis na listagem:

| Campo | Seletor / origem |
|-------|------------------|
| `lote_id` | atributo `id` do card |
| `numero_lote` + `condicao` | `<b><span>Lote N</span> - <span>CONSERVADO</span></b>` |
| `marca_modelo` | `<b>` na linha central (ex.: `HONDA/C100 BIZ 1999`) |
| `valor_atual` | `#valor_atual_lote_{id}` (formato `R$ 200,00`) |
| `url_detalhes` | `/lotes/detalhes/{lote_id}` |

> `valor_inicial` não aparece na listagem — fica `null` até scrape da página de detalhe (fase futura).

In [2]:
soup = BeautifulSoup(lotes_html, "lxml")
card = soup.select_one("div.card.listaLotes")
print(card.prettify()[:1800])

<div class="card listaLotes" id="312935">
 <span data-placement="top" data-toggle="tooltip" onclick="$(location).prop('href', '/lotes/detalhes/312935');" style="cursor: pointer !important;" title="Detalhes do Lote">
  <img alt="" class="card-img-top" src="/../Imagens/visualizar/leiloes/leilao_3416/img_312935_1.jpg"/>
 </span>
 <div class="card-body p-1 border-top">
  <!-- informações do Lote -->
  <div class="row">
   <div class="col-12">
    <b>
     <span>
      Lote 1
     </span>
     -
     <span class="">
      CONSERVADO
     </span>
    </b>
   </div>
  </div>
  <!-- Informações do lance -->
  <!-- ATUALIZAR ESSAS DIVS COM DADOS DO MAIOR LANCE -->
  <div class="row text-center">
   <!-- data hora fim -->
   <div class="col-12">
    <p id="relogio_312935">
     Aguarde
    </p>
   </div>
  </div>
  <div class="row">
   <div class="col-12 text-center" style="height: 40px;">
    <b>
     I/SHINERAY XY 50 Q 2015
    </b>
   </div>
  </div>
  <div class="row mb-2">
   <div class="co

## 3. Parser → DataFrame (página 1)

In [3]:
lotes_p1 = parse_lotes(lotes_html, leilao_id=edital.leilao_id, base_url=BASE_URL)

df = pd.DataFrame(
    [
        {
            "lote_id": l.lote_id,
            "leilao_id": l.leilao_id,
            "numero_lote": l.numero_lote,
            "condicao": l.condicao,
            "marca_modelo": l.marca_modelo,
            "valor_inicial": l.valor_inicial,
            "valor_atual": float(l.valor_atual) if l.valor_atual is not None else None,
            "url_detalhes": l.url_detalhes,
        }
        for l in lotes_p1
    ]
)

df.head(10)

,lote_id,leilao_id,numero_lote,condicao,marca_modelo,valor_inicial,valor_atual,url_detalhes
0,312935,3416,1,CONSERVADO,I/SHINERAY XY 50 Q 2015,None,200.0,https://leilao.detran.mg.gov.br/lotes/detalhes...
1,312295,3416,2,CONSERVADO,HONDA/C100 BIZ 1999,None,200.0,https://leilao.detran.mg.gov.br/lotes/detalhes...
2,312939,3416,3,CONSERVADO,HONDA/C100 BIZ 2003,None,200.0,https://leilao.detran.mg.gov.br/lotes/detalhes...
3,312313,3416,4,CONSERVADO,HONDA/CG 125 FAN 2007,None,500.0,https://leilao.detran.mg.gov.br/lotes/detalhes...
4,312941,3416,5,CONSERVADO,HONDA/CG 125 FAN 2008,None,500.0,https://leilao.detran.mg.gov.br/lotes/detalhes...
5,312946,3416,6,CONSERVADO,HONDA/CG 125 FAN KS 2010,None,500.0,https://leilao.detran.mg.gov.br/lotes/detalhes...
6,312310,3416,7,CONSERVADO,HONDA/CG 125 FAN KS 2011,None,500.0,https://leilao.detran.mg.gov.br/lotes/detalhes...
7,312927,3416,8,CONSERVADO,HONDA/CG 125 TITAN ES 2001,None,500.0,https://leilao.detran.mg.gov.br/lotes/detalhes...


## 4. Qualidade dos dados

In [4]:
print("Registros:", len(df))
print("\nNulls por coluna:\n", df.isna().sum())
print("\nDuplicatas lote_id:", df["lote_id"].duplicated().sum())
print("\nCondições:\n", df["condicao"].value_counts())
print("\nValor atual — min/median/max:", df["valor_atual"].min(), df["valor_atual"].median(), df["valor_atual"].max())

assert df["lote_id"].notna().all()
assert df["marca_modelo"].str.len().gt(0).all()
assert df["lote_id"].is_unique

Registros: 8

Nulls por coluna:
 lote_id          0
leilao_id        0
numero_lote      0
condicao         0
marca_modelo     0
valor_inicial    8
valor_atual      0
url_detalhes     0
dtype: int64

Duplicatas lote_id: 0

Condições:
 condicao
CONSERVADO    8
Name: count, dtype: int64

Valor atual — min/median/max: 200.0 500.0 500.0


## 5. Paginação — todos os lotes de um edital

In [7]:
from dataclasses import asdict

from detran_scraper.parsers import parse_lotes_from_pages

with DetranClient(base_url=BASE_URL) as client:
    pages = client.fetch_lotes_pages(lista_path)

lotes_all = parse_lotes_from_pages(pages, leilao_id=edital.leilao_id, base_url=BASE_URL)
df_all = pd.DataFrame([asdict(l) for l in lotes_all])

print(f"Páginas: {len(pages)} | Lotes totais: {len(df_all)}")
print(df_all.groupby("condicao").size())
df_all[["lote_id", "numero_lote", "marca_modelo", "valor_atual"]].head()

Páginas: 14 | Lotes totais: 111
condicao
CONSERVADO    36
SUCATA        75
dtype: int64


,lote_id,numero_lote,marca_modelo,valor_atual
0,312935,1,I/SHINERAY XY 50 Q 2015,200.00
1,312295,2,HONDA/C100 BIZ 1999,200.00
2,312939,3,HONDA/C100 BIZ 2003,200.00
3,312313,4,HONDA/CG 125 FAN 2007,500.00
4,312941,5,HONDA/CG 125 FAN 2008,500.00


## 6. Amostra de vários editais (sanity check)

Contagem de lotes por edital na home — útil para estimar volume do scrape completo.

In [8]:
rows = []
with DetranClient(base_url=BASE_URL) as client:
    for e in editais[:5]:
        path = e.url_detalhes.replace(BASE_URL, "")
        html = client.fetch(path)
        n_p1 = len(parse_lotes(html, leilao_id=e.leilao_id, base_url=BASE_URL))
        n_pages = parse_lotes_max_page(html)
        rows.append({
            "numero_edital": e.numero_edital,
            "status": e.status,
            "lotes_pagina_1": n_p1,
            "paginas": n_pages,
            "lotes_estimado": n_p1 if n_pages == 1 else None,  # preencher se quiser fetch completo
        })

pd.DataFrame(rows)

,numero_edital,status,lotes_pagina_1,paginas,lotes_estimado
0,1692/2026,Publicado,8,14,None
1,1714/2026,Publicado,8,8,None
2,1608/2026,Publicado,8,22,None
3,1690/2026,Publicado,8,3,None
4,1691/2026,Publicado,8,2,None
